In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'pyloudnorm>=0.1.1',
    'torch>=2.3.0',
    'torchaudio>=2.3.0',
    'faster-whisper>=1.0.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)


In [ ]:
import os
import re
import json
import time
import shutil
import threading
import subprocess
import sys
from pathlib import Path
from datetime import datetime, timezone

import yaml
import requests
import numpy as np
import soundfile as sf
import pyloudnorm as pyln
import torch
from huggingface_hub import HfApi, hf_hub_download

WORK_DIR        = Path('/kaggle/working')
STANDARD_DIR    = WORK_DIR / 'standardized'
SEGMENTS_DIR    = WORK_DIR / 'segments'
SNR_FLAG_DIR    = WORK_DIR / 'snr_flagged'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1c.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

STANDARD_DIR.mkdir(parents=True, exist_ok=True)
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)
SNR_FLAG_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR       = 24000
TARGET_LUFS     = -23.0
CHUNK_MINUTES   = 10

MIN_SEG_SEC     = 1.5
MAX_SEG_SEC     = 15.0
SNR_PASS        = 12.0
SNR_FLAG        = 3.0
SNR_REJECT      = 1.5

WHISPER_CONF    = 0.65
URDU_CHARS      = set('ابپتثجچحخدذرزژسشصضطظعغفقکگلمنوہھیئاآءۃے')
SAVE_EVERY      = 10
NOISE_PERCENTILE = 20

AUDIO_EXTENSIONS = {'.wav', '.flac', '.mp3', '.ogg', '.m4a', '.opus', '.webm', '.aac', '.wma'}


In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'GEMINI_API_KEY':  c.get_secret('GEMINI_API_KEY_01') or c.get_secret('GEMINI_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')
    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS     = load_secrets()
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0 repo: {STAGE0_REPO}')


In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local \u2014 done={len(state["done"])} segments={state["stats"]["total_segments"]}')
            return state
        except Exception:
            pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1c.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback \u2014 done={len(state["done"])}')
            return state
    except Exception:
        pass
    print('[checkpoint] fresh start')
    return {
        'done': [],
        'hf_deleted': [],
        'stats': {
            'total_segments': 0,
            'snr_pass': 0,
            'snr_flag': 0,
            'snr_reject': 0,
            'lang_reject': 0,
            'loudness_reject': 0,
            'loudness_peaklimited': 0,
            'loudness_passthrough': 0,
            'resampled': 0,
            'ffmpeg_recovered': 0,
            'non_wav_converted': 0,
            'stereo_downmixed': 0,
        },
        'last_updated': None,
    }

cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload:
        return
    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p1c.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1c checkpoint',
            )
            return
        except Exception as e:
            time.sleep(min(2 ** attempt, 60))

state    = load_checkpoint()
done_set = set(state['done'])
if 'hf_deleted' not in state:
    state['hf_deleted'] = []
hf_deleted_set = set(state['hf_deleted'])
for key in ('loudness_peaklimited', 'loudness_passthrough', 'resampled', 'ffmpeg_recovered', 'non_wav_converted', 'stereo_downmixed'):
    if key not in state['stats']:
        state['stats'][key] = 0


In [ ]:
print('[hf_download] listing audio files in HF repo...')
hf_audio_files = []
try:
    all_hf_files = HF_API.list_repo_files(repo_id=STAGE0_REPO, repo_type='dataset')
    hf_audio_files = [
        f for f in all_hf_files
        if f.startswith('audio/') and Path(f).suffix.lower() in AUDIO_EXTENSIONS
    ]
    hf_vid_map = {}
    for f in hf_audio_files:
        vid_id = Path(f).stem
        if vid_id not in hf_vid_map:
            hf_vid_map[vid_id] = f
        elif Path(f).suffix == '.wav':
            hf_vid_map[vid_id] = f
        elif Path(f).suffix == '.flac' and Path(hf_vid_map[vid_id]).suffix != '.wav':
            hf_vid_map[vid_id] = f
    hf_vid_ids = set(hf_vid_map.keys())
    print(f'[hf_download] {len(hf_audio_files)} audio files ({len(hf_vid_ids)} unique vid_ids) found on HF')
    ext_counts = {}
    for f in hf_audio_files:
        ext = Path(f).suffix
        ext_counts[ext] = ext_counts.get(ext, 0) + 1
    print(f'[hf_download] format breakdown: {ext_counts}')
except Exception as e:
    print(f'[hf_download] WARNING: could not list repo files: {e}')
    hf_vid_map = {}
    hf_vid_ids = set()

pending_hf_ids = sorted(hf_vid_ids - done_set - hf_deleted_set)
print(f'[hf_download] {len(done_set)} already done, {len(hf_deleted_set)} already deleted from HF, {len(pending_hf_ids)} to download')

downloaded_count  = 0
skipped_existing  = 0
failed_downloads  = []

for vid_id in pending_hf_ids:
    hf_path = hf_vid_map[vid_id]
    hf_ext  = Path(hf_path).suffix
    local_raw = STANDARD_DIR / f'{vid_id}{hf_ext}'
    local_wav = STANDARD_DIR / f'{vid_id}.wav'
    if local_wav.exists() and local_wav.stat().st_size > 0:
        skipped_existing += 1
        continue
    success = False
    for attempt in range(6):
        try:
            cached_path = hf_hub_download(
                repo_id=STAGE0_REPO,
                filename=hf_path,
                repo_type='dataset',
                token=HF_TOKEN,
            )
            shutil.copy2(cached_path, str(local_raw))
            downloaded_count += 1
            success = True
            break
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[hf_download] {vid_id} attempt {attempt+1}/6 failed: {e} \u2014 retry in {wait}s')
            time.sleep(wait)
    if not success:
        failed_downloads.append(vid_id)
        print(f'[hf_download] FAILED to download {vid_id} after 6 attempts')
        continue
    if hf_ext.lower() != '.wav':
        print(f'  [convert] {vid_id}: {hf_ext} -> .wav via ffmpeg')
        conv_result = subprocess.run(
            ['ffmpeg', '-y', '-i', str(local_raw),
             '-ar', str(TARGET_SR), '-ac', '1', '-sample_fmt', 's16', '-vn',
             str(local_wav)],
            capture_output=True, timeout=120,
        )
        if conv_result.returncode == 0 and local_wav.exists() and local_wav.stat().st_size > 0:
            local_raw.unlink(missing_ok=True)
            with cp_lock:
                state['stats']['non_wav_converted'] += 1
        else:
            print(f'  [convert] {vid_id}: ffmpeg conversion failed, trying to use raw file')
            local_raw.rename(local_wav)

print(f'\n[hf_download] summary: downloaded={downloaded_count} skipped_existing={skipped_existing} failed={len(failed_downloads)}')
if failed_downloads:
    print(f'[hf_download] failed IDs: {failed_downloads[:20]}{"..." if len(failed_downloads) > 20 else ""}')


In [ ]:
p1e_uploaded_vids = set()
try:
    url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1e.json'
    r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
    if r.status_code == 200:
        p1e_state = r.json()
        for seg_id in p1e_state.get('uploaded_ids', []):
            vid_id = seg_id.split('_c')[0]
            p1e_uploaded_vids.add(vid_id)
        p1e_source_deleted = set(p1e_state.get('source_deleted', []))
        hf_deleted_set.update(p1e_source_deleted - hf_deleted_set)
        print(f'[hf_delete] p1e reports {len(p1e_uploaded_vids)} vid_ids with uploaded segments, {len(p1e_source_deleted)} already deleted by p1e')
    else:
        print(f'[hf_delete] p1e checkpoint not found on HF (status {r.status_code}) \u2014 skipping deletion')
except Exception as e:
    print(f'[hf_delete] WARNING: could not load p1e checkpoint: {e} \u2014 skipping deletion')

eligible_for_deletion = sorted((done_set & p1e_uploaded_vids) - hf_deleted_set)
print(f'[hf_delete] {len(eligible_for_deletion)} source audio files eligible for deletion from HF')

deleted_count = 0
for vid_id in eligible_for_deletion:
    for ext in AUDIO_EXTENSIONS:
        hf_file_path = f'audio/{vid_id}{ext}'
        for attempt in range(6):
            try:
                HF_API.delete_file(
                    path_in_repo=hf_file_path,
                    repo_id=STAGE0_REPO,
                    repo_type='dataset',
                )
                deleted_count += 1
                break
            except Exception as e:
                err_str = str(e)
                if '404' in err_str or 'not found' in err_str.lower():
                    break
                wait = min(2 ** attempt, 60)
                print(f'[hf_delete] {vid_id}{ext} attempt {attempt+1}/6 failed: {e} \u2014 retry in {wait}s')
                time.sleep(wait)
    hf_deleted_set.add(vid_id)
    if vid_id not in state['hf_deleted']:
        state['hf_deleted'].append(vid_id)

if deleted_count > 0:
    save_checkpoint(state, upload=True)
    print(f'[hf_delete] deleted {deleted_count} source files from HF (total deleted: {len(hf_deleted_set)})')
elif eligible_for_deletion:
    print(f'[hf_delete] failed to delete any of {len(eligible_for_deletion)} eligible files')
else:
    print('[hf_delete] no eligible files to delete')


In [ ]:
vad_model_cache = {}

def get_vad_model():
    if 'model' not in vad_model_cache:
        model, utils = torch.hub.load(
            repo_or_dir='snakers4/silero-vad',
            model='silero_vad',
            force_reload=False,
            onnx=False,
            verbose=False,
        )
        vad_model_cache['model'] = model
        vad_model_cache['get_ts'] = utils[0]
        print('[vad] Silero VAD loaded')
    return vad_model_cache['model'], vad_model_cache['get_ts']

whisper_cache = {}

def get_whisper(size='tiny', device='cpu'):
    key = (size, device)
    if key not in whisper_cache:
        from faster_whisper import WhisperModel
        compute = 'float16' if device == 'cuda' else 'int8'
        whisper_cache[key] = WhisperModel(size, device=device, compute_type=compute)
        print(f'[whisper] {size} loaded on {device}')
    return whisper_cache[key]

def compute_snr(audio, frame_length=2048, noise_percentile=None):
    if noise_percentile is None:
        noise_percentile = NOISE_PERCENTILE
    if len(audio) < frame_length:
        return 0.0
    hop = frame_length // 2
    energies = np.array([
        np.mean(audio[i:i + frame_length] ** 2)
        for i in range(0, len(audio) - frame_length, hop)
    ])
    energies = energies[energies > 0]
    if len(energies) == 0:
        return 0.0
    noise_floor = np.percentile(energies, noise_percentile)
    if noise_floor <= 0:
        return 60.0
    return float(10 * np.log10(np.mean(energies) / noise_floor))

def snr_label(snr_db):
    if snr_db >= SNR_PASS:
        return 'pass'
    if snr_db >= SNR_FLAG:
        return 'flag'
    if snr_db >= SNR_REJECT:
        return 'flag'
    return 'reject'

def normalize_loudness(audio, sr):
    audio_f64 = audio.astype(np.float64)
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(audio_f64)
    if not np.isfinite(loudness):
        peak = np.max(np.abs(audio_f64))
        if peak > 1e-6:
            normalized = audio_f64 * (0.5 / peak)
            return normalized.astype(np.float32), False
        return None
    normalized = pyln.normalize.loudness(audio_f64, loudness, TARGET_LUFS)
    peak = np.max(np.abs(normalized))
    if peak > 0.99:
        normalized = normalized * (0.99 / peak)
        return normalized.astype(np.float32), True
    return normalized.astype(np.float32), False

def detect_language_chunk(audio_path):
    model = get_whisper('tiny', 'cpu')
    _, info = model.transcribe(str(audio_path), language=None, task='transcribe', beam_size=1)
    lang = info.language
    prob = round(info.language_probability, 3)
    if lang == 'hi' and prob < 0.85:
        lang = 'ur'
    return lang, prob

def repair_wav_with_ffmpeg(input_path, output_path=None):
    if output_path is None:
        output_path = input_path.with_suffix('.repaired.wav')
    try:
        result = subprocess.run(
            ['ffmpeg', '-y', '-i', str(input_path),
             '-ar', str(TARGET_SR), '-ac', '1', '-sample_fmt', 's16', '-vn',
             str(output_path)],
            capture_output=True, timeout=120,
        )
        if result.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0:
            return output_path
    except Exception:
        pass
    return None

def run_vad_on_chunk(audio_array, sr=TARGET_SR):
    model, get_ts = get_vad_model()
    tensor = torch.FloatTensor(audio_array)
    raw = get_ts(tensor, model, sampling_rate=sr)
    segments = []
    for ts in raw:
        start = ts['start'] / sr
        end   = ts['end']   / sr
        dur   = end - start
        if dur < MIN_SEG_SEC:
            continue
        if dur <= MAX_SEG_SEC:
            segments.append({'start': start, 'end': end, 'duration': round(dur, 3)})
            continue
        cursor = start
        while cursor < end:
            seg_end = min(cursor + MAX_SEG_SEC, end)
            seg_dur = seg_end - cursor
            if seg_dur >= MIN_SEG_SEC:
                segments.append({'start': cursor, 'end': seg_end, 'duration': round(seg_dur, 3)})
            cursor += MAX_SEG_SEC
    return segments

def ensure_mono(audio):
    if audio.ndim == 1:
        return audio, False
    if audio.ndim == 2:
        mono = np.mean(audio, axis=1)
        return mono.astype(np.float32), True
    return audio.flatten(), True


In [ ]:
manifest_local = WORK_DIR / 'video_manifest.jsonl'
video_meta     = {}
if manifest_local.exists():
    with open(manifest_local, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                v = json.loads(line)
                video_meta[v['video_id']] = v
wav_files = sorted(STANDARD_DIR.glob('*.wav'))
pending   = [p for p in wav_files if p.stem not in done_set]
print(f'[clean_cpu] {len(wav_files)} total WAVs, {len(done_set)} already done, {len(pending)} to process')
get_vad_model()
get_whisper('tiny', 'cpu')


In [ ]:
seg_records = []

for file_idx, wav_path in enumerate(pending):
    vid_id   = wav_path.stem
    vid_info = video_meta.get(vid_id, {})
    file_segs_written = 0
    try:
        try:
            with sf.SoundFile(str(wav_path)) as sf_file:
                total_samples = len(sf_file)
                sr            = sf_file.samplerate
                channels      = sf_file.channels
        except Exception as sf_err:
            print(f'  [repair] {vid_id}: soundfile failed ({sf_err}), trying ffmpeg repair')
            repaired = repair_wav_with_ffmpeg(wav_path)
            if repaired is None:
                raise RuntimeError(f'soundfile + ffmpeg both failed: {sf_err}')
            repaired.replace(wav_path)
            with sf.SoundFile(str(wav_path)) as sf_file:
                total_samples = len(sf_file)
                sr            = sf_file.samplerate
                channels      = sf_file.channels
            with cp_lock:
                state['stats']['ffmpeg_recovered'] += 1
            print(f'  [repair] {vid_id}: ffmpeg recovery succeeded')
        total_duration = total_samples / sr
        if sr != TARGET_SR:
            print(f'  [resample] {vid_id}: {sr}Hz -> {TARGET_SR}Hz')
            resampled_path = wav_path.with_suffix('.resampled.wav')
            resample_result = subprocess.run(
                ['ffmpeg', '-y', '-i', str(wav_path),
                 '-ar', str(TARGET_SR), '-ac', '1', '-sample_fmt', 's16', '-vn',
                 str(resampled_path)],
                capture_output=True, timeout=120,
            )
            if resample_result.returncode == 0 and resampled_path.exists():
                resampled_path.replace(wav_path)
                sr = TARGET_SR
                channels = 1
                with sf.SoundFile(str(wav_path)) as sf_file:
                    total_samples = len(sf_file)
                with cp_lock:
                    state['stats']['resampled'] += 1
            else:
                print(f'  [resample] {vid_id}: ffmpeg resample failed, skipping')
                raise RuntimeError(f'failed to resample {vid_id} from {sr}Hz')
        chunk_samples  = CHUNK_MINUTES * 60 * sr
        chunk_offset   = 0
        chunk_idx      = 0
        while chunk_offset < total_samples:
            with sf.SoundFile(str(wav_path)) as sf_file:
                sf_file.seek(chunk_offset)
                chunk = sf_file.read(chunk_samples, dtype='float32')
            if len(chunk) == 0:
                break
            chunk, was_stereo = ensure_mono(chunk)
            if was_stereo:
                with cp_lock:
                    state['stats']['stereo_downmixed'] += 1
            chunk_offset_sec = chunk_offset / sr
            vad_segs         = run_vad_on_chunk(chunk, sr)
            for seg in vad_segs:
                abs_start = chunk_offset_sec + seg['start']
                abs_end   = chunk_offset_sec + seg['end']
                start_idx = int(seg['start'] * sr)
                end_idx   = int(seg['end']   * sr)
                seg_audio = chunk[start_idx:end_idx]
                snr_db  = compute_snr(seg_audio)
                s_label = snr_label(snr_db)
                with cp_lock:
                    state['stats']['total_segments'] += 1
                    state['stats'][f'snr_{s_label}'] += 1
                if s_label == 'reject':
                    continue
                result = normalize_loudness(seg_audio, sr)
                if result is None:
                    with cp_lock:
                        state['stats']['loudness_reject'] += 1
                    continue
                normalized, was_peak_limited = result
                if was_peak_limited:
                    with cp_lock:
                        state['stats']['loudness_peaklimited'] += 1
                seg_id   = f'{vid_id}_c{chunk_idx:03d}_s{file_segs_written:04d}'
                seg_file = SEGMENTS_DIR / f'{seg_id}.wav' if s_label == 'pass' else SNR_FLAG_DIR / f'{seg_id}.wav'
                sf.write(str(seg_file), normalized, sr, subtype='PCM_16')
                seg_records.append({
                    'seg_id':          seg_id,
                    'video_id':        vid_id,
                    'video_title':     vid_info.get('title', ''),
                    'channel_id':      vid_info.get('channel_id', ''),
                    'channel_category': vid_info.get('category', 'general'),
                    'query_used':      vid_info.get('query_used', ''),
                    'abs_start_sec':   round(abs_start, 3),
                    'abs_end_sec':     round(abs_end, 3),
                    'duration_sec':    round(seg['duration'], 3),
                    'snr_db':          round(snr_db, 2),
                    'snr_label':       s_label,
                    'peak_limited':    was_peak_limited,
                    'seg_path':        str(seg_file),
                })
                file_segs_written += 1
            chunk_offset += chunk_samples
            chunk_idx    += 1
        with cp_lock:
            if vid_id not in done_set:
                done_set.add(vid_id)
                state['done'].append(vid_id)
    except Exception as e:
        print(f'  [error] {vid_id}: {e}')
    if (file_idx + 1) % SAVE_EVERY == 0 or file_idx + 1 == len(pending):
        upload_now = (file_idx + 1) % (SAVE_EVERY * 5) == 0
        save_checkpoint(state, upload=upload_now)
        print(f'  [{file_idx+1}/{len(pending)}] segs={state["stats"]["total_segments"]} '
              f'pass={state["stats"]["snr_pass"]} flag={state["stats"]["snr_flag"]} '
              f'reject={state["stats"]["snr_reject"]} '
              f'peak_limited={state["stats"]["loudness_peaklimited"]} '
              f'stereo_downmix={state["stats"]["stereo_downmixed"]}')


In [ ]:
seg_records_path = WORK_DIR / 'seg_records_pre_lang.jsonl'
with open(seg_records_path, 'w', encoding='utf-8') as f:
    for rec in seg_records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f'\n[clean_cpu] summary')
print(f'  total segments     : {state["stats"]["total_segments"]}')
print(f'  snr pass           : {state["stats"]["snr_pass"]}')
print(f'  snr flag (demucs)  : {state["stats"]["snr_flag"]}')
print(f'  snr reject         : {state["stats"]["snr_reject"]}')
print(f'  loudness reject    : {state["stats"]["loudness_reject"]}')
print(f'  loudness peak-lim  : {state["stats"]["loudness_peaklimited"]}')
print(f'  resampled          : {state["stats"]["resampled"]}')
print(f'  ffmpeg recovered   : {state["stats"]["ffmpeg_recovered"]}')
print(f'  non-wav converted  : {state["stats"]["non_wav_converted"]}')
print(f'  stereo downmixed   : {state["stats"]["stereo_downmixed"]}')
print(f'  records written    : {len(seg_records)}')
print(f'  pass segments dir  : {SEGMENTS_DIR}')
print(f'  flag segments dir  : {SNR_FLAG_DIR}')
print(f'  HF source deleted  : {len(hf_deleted_set)}')
save_checkpoint(state, upload=True)
print('\n[done] ready for p1d_clean_gpu.ipynb')
